In [ ]:
import numpy as np
from scipy.stats import linregress

rng = np.random.default_rng(0)
n_boot = 2000

raw_df = combined_data[uniprot_id][["deltaG", "ddg_rebased", "variant_rebased"]].dropna()
median_df = combined_data_median_ex[uniprot_id][["deltaG", "ddg_rebased"]].dropna()

def bootstrap_slope(df, value_col="ddg_rebased", n_boot=n_boot):
    slopes = []
    n = len(df)
    x = df["deltaG"].to_numpy()
    y = df[value_col].to_numpy()
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(set(x[idx])) < 2:
            continue
        slope, *_ = linregress(x[idx], y[idx])
        slopes.append(slope)
    slopes = np.array(slopes)
    return slopes, np.percentile(slopes, [2.5, 97.5])

raw_slopes, raw_ci = bootstrap_slope(raw_df)
median_slopes, median_ci = bootstrap_slope(median_df)

print(f"{gene} raw slope: point estimate {linregress(raw_df['deltaG'], raw_df['ddg_rebased']).slope:.3f}, "
      f"95% bootstrap CI [{raw_ci[0]:.3f}, {raw_ci[1]:.3f}], n_variants={raw_df['variant_rebased'].nunique()}")
print(f"{gene} median-aggregated slope: point estimate {linregress(median_df['deltaG'], median_df['ddg_rebased']).slope:.3f}, "
      f"95% bootstrap CI [{median_ci[0]:.3f}, {median_ci[1]:.3f}], n_variants={len(median_df)}")

overlap = not (raw_ci[1] < median_ci[0] or median_ci[1] < raw_ci[0])
print(f"\nCIs overlap: {overlap}  -> "
      f"{'improvement is NOT clearly distinguishable from noise at this n' if overlap else 'improvement appears robust, CIs do not overlap'}")

In [ ]:
# Take the mean ddg to average across homomeric chains in pipeline structures SPG/FYN **DELETE**

pipeline_data_meanhomo = defaultdict(dict)

for uniprot_id, df in pipeline_data.items():
    # Group by pdb_id + variant to take mean ddg
    for pdb_id, pdb_df in df.groupby("pdb_id"):
        mean_df = (
            pdb_df.groupby(["uniprot_id", "pdb_id", "variant_rebased"], as_index=False)
                  .agg({
                      "ddg": "mean",                 # raw ddg instead of ddg_rebased
                      "mut_from_rebased": "first",
                      "mut_to": "first",
                      "res": "first",
                  })
                  .rename(columns={"ddg": "ddg_rebased"})   # keep the name the rest of the notebook uses
        )
        pipeline_data_meanhomo[uniprot_id][pdb_id] = mean_df


In [ ]:
def fit_fn_median_plain(x):
    slope, intercept, r_value, p_value, std_err = linregress(x, combined_data_median[uniprot_id]['ddg_rebased'])
    print(f"{gene} Slope medians (no outlier removal) = {slope:.3f} ± {std_err:.3f}")
    return slope * x + intercept

plt.figure(figsize=(8, 8), dpi=300)
plt.rcParams.update({'font.size': 18})

# Plot all energies
plt.scatter(combined_data[uniprot_id]['deltaG'], combined_data[uniprot_id]['ddg_rebased'],
            label='ΔΔG', color='dodgerblue', alpha=0.5)

# Plot median (no outlier removal — insufficient coverage for reliable outlier ID)
plt.scatter(combined_data_median[uniprot_id]['deltaG'], combined_data_median[uniprot_id]['ddg_rebased'],
            label='median ΔΔG', color='black')

# Plot LOBF line for plain median
plt.plot(
    combined_data_median[uniprot_id]['deltaG'],
    fit_fn_median_plain(combined_data_median[uniprot_id]['deltaG']),
    '--k', label='LOBF for medians'
)

plt.xlim(plt.xlim()[1], plt.xlim()[0])  # reverse x-axis as Tsuboyama present an 'unfolded state model' and so energies are unfolding
plt.xlabel('Tsuboyama et al. Experimental ΔΔG (kcal/mol)')
plt.ylabel('FoldX Pipeline ΔΔG (kcal/mol)')
plt.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)
plt.tight_layout()
plt.savefig("../Figures/Figure_4d.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def fit_fn_median_plain(x):
    slope, intercept, r_value, p_value, std_err = linregress(x, combined_data_median[uniprot_id]['ddg_rebased'])
    print(f"{gene} Slope medians (no outlier removal) = {slope:.3f} ± {std_err:.3f}")
    return slope * x + intercept

plt.figure(figsize=(8, 8), dpi=300)
plt.rcParams.update({'font.size': 18})


# Plot all raw per-structure energies (context only, not aggregated)
plt.scatter(combined_data[uniprot_id]['deltaG'], combined_data[uniprot_id]['ddg_rebased'],
            label='ΔΔG', color='dodgerblue', alpha=0.5)

# Plot median ΔΔG, no outlier removal
plt.scatter(combined_data_median[uniprot_id]['deltaG'], combined_data_median[uniprot_id]['ddg_rebased'],
            label='median ΔΔG', color='black')

# Line of best fit for the plain median
plt.plot(
    combined_data_median[uniprot_id]['deltaG'],
    fit_fn_median_plain(combined_data_median[uniprot_id]['deltaG']),
    '--k', label='LOBF for medians'
)

plt.xlim(plt.xlim()[1], plt.xlim()[0])  # reverse x-axis, as before
plt.xlabel('Tsuboyama et al. Experimental ΔΔG (kcal/mol)')
plt.ylabel('FoldX Pipeline ΔΔG (kcal/mol)')
plt.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

#plt.title(f'{gene}: median-aggregated ΔΔG (no outlier removal)')
plt.savefig("../Figures/Figure_4b.png", dpi=300, bbox_inches="tight")

plt.show()
